# Does hard-positive mining help?

Three fine-tunes of `solar_unet_ms.pt`, identical except for what is in the
training set:

| arm | train crops | mined | question |
|---|---|---|---|
| `jbeil06r_seg` | 850 | 0% | baseline |
| `jbeil06r_pos4` | 1,190 | 29% | does oversampling the missed arrays help? |
| `jbeil06r_pos` | 1,521 | 44% | does more of it help, or start costing precision? |

**What is being tested.** 115 verified arrays at Jbeil get a peak model
response of 0.013, against 0.982 for the ones the detector finds. Most were
already in the training set with correct masks, so simply including them has
been tried and failed. The hypothesis is that they are *drowned*: a 30 m2
array is ~3% of a 512 px crop, so across 850 crops the whole missed population
is a fraction of a percent of the loss and the cheapest thing the optimiser can
do is call them background. `mine_hard_positives.py` re-cuts crops centred on
each, so they carry weight.

**Expect the baseline arm to be flat.** `solar_unet_ms.pt` was already
fine-tuned on this region; 7 epochs locally moved F1 from 0.443 to 0.438. The
mined arms have to beat that flat line, not zero.

**Runtime:** ~5 minutes per arm on a T4. Set Runtime > Change runtime type > T4 GPU.

## 0 - Preflight

Colab and Kaggle differ in writable paths, in how files get in, and in how they
come out, so that is resolved once here rather than sprinkled through the
notebook. Same pattern as `train_multiscale.ipynb`.

- **Kaggle:** Settings > Accelerator > **GPU T4 x2**. Attach the data as a
  Dataset (Add Input); it lands read-only under `/kaggle/input/`.
- **Colab:** Runtime > Change runtime type > **T4 GPU**. Upload in the next
  cell, or mount Drive.


In [ ]:
import sys, os, shutil, glob, zipfile
from pathlib import Path

if Path('/kaggle').exists():
    ENV  = 'kaggle'
    WORK = Path('/kaggle/temp/solarmap')   # scratch: big, not preserved
    OUT  = Path('/kaggle/working')         # appears in the Output tab
    ROOTS = [Path('/kaggle/input')]
elif 'google.colab' in sys.modules or Path('/content').exists():
    ENV  = 'colab'
    WORK = OUT = Path('/content')
    ROOTS = [Path('/content'), Path('/content/drive/MyDrive')]
else:
    ENV  = 'local'
    WORK = OUT = Path('./work')
    ROOTS = [Path('.')]
WORK.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
DATA = WORK / 'datasets'; DATA.mkdir(parents=True, exist_ok=True)

import torch
GPU = torch.cuda.is_available()
print(f'environment : {ENV}')
print(f'scratch     : {WORK}')
print(f'outputs     : {OUT}')
print('CUDA        :', GPU, '|', torch.cuda.get_device_name(0) if GPU else 'NO GPU')
if not GPU:
    print('              -> Kaggle: Settings > Accelerator > GPU T4 x2')
    print('              -> Colab:  Runtime > Change runtime type > T4 GPU')
print(f'free disk   : {shutil.disk_usage(WORK).free/1e9:.0f} GB')


## 1 - Get the data in

Four inputs are needed:

| file | size | what it is |
|---|---|---|
| `jbeil06r_seg.zip` | 182 MB | baseline train/val split |
| `jbeil06r_pos4_mined.zip` | 62 MB | mined crops only, 29% arm |
| `jbeil06r_pos_mined.zip` | 122 MB | mined crops only, 44% arm |
| `solar_unet_ms.pt` | 93 MB | starting weights |

The mined zips hold only the `hardpos_*` crops. Each arm is rebuilt below as
base + mined, so the 850 baseline crops travel once rather than three times.

**Kaggle** (better for this): make a Dataset from the four files once, then Add
Input on any future run - no re-upload, and 30 GPU-hours a week. Note Kaggle
**auto-extracts** zips when building a Dataset, so the finder below accepts
either a `.zip` or an already-extracted folder.

**Colab:** put the files in Drive and mount it, or let the cell prompt for an
upload. The browser uploader is unreliable above ~100 MB and `jbeil06r_seg.zip`
is 182 MB, so Drive is the safer route.


In [ ]:
def find(stem, marker=None):
    """Locate an input as a zip, or as an already-extracted directory.

    Kaggle unpacks zips when a Dataset is created, so the same input arrives as
    a folder there and as a .zip on Colab. Accepting both is what lets one
    notebook run on either.

    The name test is EXACT, not a substring: 'jbeil06r_pos' is a substring of
    'jbeil06r_pos4_mined', so substring matching silently handed the 44% arm
    the 29% arm's crops, and both arms would have looked identical for no
    visible reason.
    """
    for root in ROOTS:
        if root.exists():
            for q in root.rglob(stem + '.zip'):
                return ('zip', q)
    base = stem.replace('_mined', '')
    for root in ROOTS:
        if not root.exists():
            continue
        for q in root.rglob('*'):
            if not q.is_dir() or q.name not in (stem, base):
                continue
            if marker and not list(q.glob(marker)):
                continue
            return ('dir', q)
    return (None, None)

def materialise(kind, src, dest):
    dest.mkdir(parents=True, exist_ok=True)
    if kind == 'zip':
        with zipfile.ZipFile(src) as z:
            z.extractall(dest)
    else:
        shutil.copytree(src, dest, dirs_exist_ok=True)

# On Colab with nothing staged, fall back to the uploader.
if ENV == 'colab' and find('jbeil06r_seg', 'train')[0] is None:
    from google.colab import files
    print('Select ALL FOUR files at once (ctrl-click):')
    files.upload()

kind, src = find('jbeil06r_seg', 'train')
assert kind, 'jbeil06r_seg not found - attach the Dataset / upload it first'
materialise(kind, src, DATA / 'jbeil06r_seg')

for arm, stem in (('jbeil06r_pos4', 'jbeil06r_pos4_mined'),
                  ('jbeil06r_pos',  'jbeil06r_pos_mined')):
    k, s = find(stem, 'train/images/hardpos_*.png')
    assert k, stem + ' not found'
    if (DATA / arm).exists():
        shutil.rmtree(DATA / arm)
    shutil.copytree(DATA / 'jbeil06r_seg', DATA / arm)
    materialise(k, s, DATA / arm)

WEIGHTS = None
for root in ROOTS:
    if root.exists():
        for q in root.rglob('solar_unet_ms.pt'):
            WEIGHTS = q
            break
    if WEIGHTS:
        break
assert WEIGHTS, 'solar_unet_ms.pt not found'
print('weights:', WEIGHTS)
print()

for arm in ('jbeil06r_seg', 'jbeil06r_pos4', 'jbeil06r_pos'):
    tr = len(glob.glob(str(DATA / arm / 'train' / 'images' / '*.png')))
    mi = len(glob.glob(str(DATA / arm / 'train' / 'images' / 'hardpos_*.png')))
    va = len(glob.glob(str(DATA / arm / 'val' / 'images' / '*.png')))
    print(f'{arm:16} train={tr:5}  mined={mi:4} ({100*mi/max(tr,1):2.0f}%)  val={va}')

print()
print('EXPECTED:  seg 850/0 (0%)   pos4 1190/340 (29%)   pos 1521/671 (44%)')
print('If these do not match, stop - the arms did not rebuild, and training')
print('them would measure nothing.')


In [ ]:
!pip -q install segmentation-models-pytorch albumentations
import torch, segmentation_models_pytorch as smp
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 2 - Training pipeline

Kept identical to `scripts/finetune_unet.py` so the result transfers back to
the repo unchanged: 256 px crops, scale augmentation spanning roughly 5-20 cm,
a mask-aware crop half the time (a plain random crop at scale 2.0 loses the
array - measured 49.7% survival against 72.7% with this), Dice+BCE, and lr
3e-5 so the base checkpoint is adapted rather than overwritten.

**Selection is on array F1, not validation IoU.** IoU rewards tracing an array
the model already finds more tightly; the goal here is finding arrays it
currently misses. The matching rule is the one in `score_arrays.py`.

In [ ]:
import cv2, numpy as np, torch.nn as nn, albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset

TILE = 256
SCALE_LO, SCALE_HI = 0.5, 2.0

def build_tf(train, mean, std):
    if train:
        stages = [
            A.RandomScale(scale_limit=(SCALE_LO - 1.0, SCALE_HI - 1.0), p=1.0),
            A.PadIfNeeded(TILE, TILE, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
            A.OneOf([A.CropNonEmptyMaskIfExists(TILE, TILE, p=1.0),
                     A.RandomCrop(TILE, TILE, p=1.0)], p=1.0),
            A.HorizontalFlip(p=.5), A.VerticalFlip(p=.5), A.RandomRotate90(p=1.),
            A.RandomBrightnessContrast(.25, .25, p=.7),
            A.HueSaturationValue(10, 20, 12, p=.4),
        ]
    else:
        stages = [A.PadIfNeeded(TILE, TILE, border_mode=cv2.BORDER_REFLECT_101, p=1.0),
                  A.CenterCrop(TILE, TILE, p=1.0)]
    return A.Compose(stages + [A.Normalize(mean=mean, std=std), ToTensorV2()])

class SegDS(Dataset):
    def __init__(self, root, split, mean, std):
        self.imgs = sorted((Path(root) / split / 'images').glob('*'))
        self.mdir = Path(root) / split / 'masks'
        self.tf = build_tf(split == 'train', mean, std)
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        p = self.imgs[i]
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        m = cv2.imread(str(self.mdir / p.name), cv2.IMREAD_GRAYSCALE)
        m = np.zeros(img.shape[:2], np.uint8) if m is None else m
        o = self.tf(image=img, mask=(m > 127).astype(np.float32))
        return o['image'], o['mask'].unsqueeze(0)

class DiceBCE(nn.Module):
    def __init__(self, w=.5):
        super().__init__(); self.w = w; self.bce = nn.BCEWithLogitsLoss()
    def forward(self, logits, t):
        p = torch.sigmoid(logits)
        num = 2 * (p * t).sum((1,2,3)) + 1
        den = p.sum((1,2,3)) + t.sum((1,2,3)) + 1
        return self.w * self.bce(logits, t) + (1 - self.w) * (1 - (num / den).mean())

@torch.no_grad()
def array_scores(model, loader, dev, thr=0.5, min_cover=.5, min_on=.5):
    located = total = tp = fp = 0
    model.eval()
    for x, y in loader:
        pr = (torch.sigmoid(model(x.to(dev))) > thr).cpu().numpy()[:, 0]
        gt = y.numpy()[:, 0] > .5
        for p, g in zip(pr, gt):
            n, lab = cv2.connectedComponents(g.astype(np.uint8))
            for i in range(1, n):
                m = lab == i
                if m.sum() < 4: continue
                total += 1
                located += bool((m & p).sum() / m.sum() >= min_cover)
            n2, lab2 = cv2.connectedComponents(p.astype(np.uint8))
            for i in range(1, n2):
                m = lab2 == i
                if m.sum() < 4: continue
                if (m & g).sum() / m.sum() >= min_on: tp += 1
                else: fp += 1
    rec = float(located / max(total, 1)); prec = float(tp / max(tp + fp, 1))
    return rec, prec, float(2 * prec * rec / max(prec + rec, 1e-9)), int(total)

## 3 - Run all three arms

Same seed, same epochs, same starting weights. The only difference between
arms is the training set.

In [ ]:
import time, json

EPOCHS, BATCH, LR = 20, 16, 3e-5
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = {}

def run(arm):
    torch.manual_seed(11); np.random.seed(11)
    ck = torch.load(WEIGHTS, map_location='cpu', weights_only=False)
    mean, std = tuple(ck['mean']), tuple(ck['std'])
    model = smp.Unet(ck['encoder'], encoder_weights=None, in_channels=3, classes=1)
    model.load_state_dict(ck['state_dict']); model.to(dev)

    root = DATA / arm
    tr = DataLoader(SegDS(root, 'train', mean, std), BATCH, shuffle=True,
                    num_workers=2, drop_last=True)
    va = DataLoader(SegDS(root, 'val', mean, std), BATCH, num_workers=2)

    crit = DiceBCE(); opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scaler = torch.amp.GradScaler('cuda', enabled=(dev.type == 'cuda'))

    r0, p0, f0, n0 = array_scores(model, va, dev)
    print(f'{arm}: before  recall={r0:.3f} precision={p0:.3f} F1={f0:.3f}  ({n0} val arrays)')
    best, hist = -1.0, [{'epoch': 0, 'recall': r0, 'precision': p0, 'f1': f0}]

    for ep in range(1, EPOCHS + 1):
        model.train(); tot = 0.0; t0 = time.time()
        for x, y in tr:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', enabled=(dev.type == 'cuda')):
                loss = crit(model(x), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item()
        r, p, f, _ = array_scores(model, va, dev)
        hist.append({'epoch': ep, 'recall': r, 'precision': p, 'f1': f})
        mark = ''
        if f > best:
            best = f; mark = '  <- saved'
            torch.save({'state_dict': model.state_dict(), 'encoder': ck['encoder'],
                        'tile_size': ck.get('tile_size', 512), 'mean': list(mean),
                        'std': list(std), 'array_f1': f, 'epoch': ep, 'arm': arm},
                       str(OUT / (arm + '.pt')))
        print(f'  ep {ep:2d}  loss={tot/max(len(tr),1):.4f}  '
              f'recall={r:.3f} precision={p:.3f} F1={f:.3f}  {time.time()-t0:.0f}s{mark}')
    results[arm] = {'before': f0, 'best': best, 'history': hist}
    return best

for arm in ('jbeil06r_seg', 'jbeil06r_pos4', 'jbeil06r_pos'):
    run(arm); print()

json.dump(results, open(OUT / 'mining_results.json', 'w'), indent=1)

## 4 - Verdict

The comparison that matters is each arm's best array F1 against the baseline's,
and against the `before` line - the score of the checkpoint they all started
from. An arm that fails to beat `before` has not learned anything useful.

**This val split is small** (12 tiles). It will show a large effect and will
not resolve a small one, so treat a difference under a couple of points as
noise rather than a result. The real test is `score_arrays.py` on the capture,
restricted to the held-out tiles, once the checkpoints are back in `models/`.

In [ ]:
print(f"{'arm':16}{'mined':>8}{'before':>9}{'best F1':>10}{'delta':>9}")
base = results['jbeil06r_seg']['best']
for arm, share in (('jbeil06r_seg', '0%'), ('jbeil06r_pos4', '29%'), ('jbeil06r_pos', '44%')):
    r = results[arm]
    print(f"{arm:16}{share:>8}{r['before']:>9.3f}{r['best']:>10.3f}{r['best']-base:>+9.3f}")


## 5 - Collect the checkpoints

**Kaggle:** they are in `/kaggle/working` - take them from the **Output** panel
on the right.

**Colab:** the cell below downloads them.


In [ ]:
print('files in', OUT)
for q in sorted(Path(OUT).glob('*')):
    if q.is_file():
        print(f'  {q.name:28} {q.stat().st_size/1e6:8.1f} MB')

if ENV == 'colab':
    from google.colab import files
    for arm in ('jbeil06r_seg', 'jbeil06r_pos4', 'jbeil06r_pos'):
        f = OUT / (arm + '.pt')
        if f.exists():
            files.download(str(f))
    files.download(str(OUT / 'mining_results.json'))
elif ENV == 'kaggle':
    print()
    print('Download from the Output panel on the right ->')


## 6 - Back in the repo

Drop the `.pt` files into `models/`, then get the number that actually counts -
array recall on the capture, on tiles no arm trained on:

```powershell
python scripts/detect.py --capture jbeil-mb-104 --checkpoint jbeil06r_pos.pt
python scripts/score_arrays.py --capture jbeil-mb-104 --labels labels_reviewed.json
```

The val tiles are recoverable from the crop filenames under
`data/datasets/jbeil06r_seg/val/images` - each is named `{tile_id}_{ox}_{oy}`.
Scoring the whole capture mixes trained and held-out ground and will read
several points high.
